In [ ]:
# DNABERT + CNN :
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, matthews_corrcoef, roc_auc_score
from collections import Counter
import random
import re
from tqdm import tqdm

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Data Utilities ---
def clean_sequence(seq):
    seq = seq.upper()
    return re.sub('[^ATCG]', '', seq)

def load_fasta_clean(file_path):
    with open(file_path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    complement = str.maketrans("ATCG", "TAGC")
    return seq.translate(complement)[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

# --- Shared Dataset Class ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        encoding = self.tokenizer(self.sequences[idx], padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.float),
        }

# --- DNA BERT + CNN Model ---
class DNABERT_CNN(nn.Module):
    def __init__(self, dropout_rate=0.3, kernel_size=1, num_filters=64):
        super().__init__()
        # Load pre-trained DNA BERT model
        self.bert = AutoModel.from_pretrained("zhihan1996/DNA_bert_6")
        
        # CNN Layer with smaller kernel size
        self.conv1 = nn.Conv1d(in_channels=self.bert.config.hidden_size, out_channels=num_filters, kernel_size=kernel_size)
        self.pool = nn.AdaptiveMaxPool1d(1)  # Global max pooling
        
        # Fully connected layer
        self.fc1 = nn.Linear(num_filters, 1)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self, input_ids, attention_mask):
        # Pass through BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Get the pooled output (CLS token representation)
        x = outputs.pooler_output
        
        # Reshape to fit CNN (batch_size, channels, sequence_length)
        x = x.unsqueeze(2)  # Add sequence length dimension (batch_size, 768, 1)
        
        # Apply convolution
        x = self.conv1(x)   # Apply convolution
        x = self.pool(x)    # Global max pooling
        x = x.squeeze(2)    # Squeeze sequence dimension
        
        # Pass through dropout and fully connected layer
        x = self.dropout(x)
        x = self.fc1(x)
        
        return x

# --- Evaluation Utility ---
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    tp, tn, fp, fn = 0, 0, 0, 0  # Initialize confusion matrix counts
    all_probs = []  # List to store all probability values

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)
            logits = model(input_ids, attention_mask).squeeze()
            probs = torch.sigmoid(logits)
            preds.extend((probs > 0.5).int().cpu().tolist())
            trues.extend(label.int().cpu().tolist())
            all_probs.extend(probs.cpu().tolist())  # Collect all probs

            # Update confusion matrix
            for true, pred in zip(label.int().cpu().tolist(), (probs > 0.5).int().cpu().tolist()):
                if true == 1 and pred == 1:
                    tp += 1  # True Positive
                elif true == 0 and pred == 0:
                    tn += 1  # True Negative
                elif true == 0 and pred == 1:
                    fp += 1  # False Positive
                elif true == 1 and pred == 0:
                    fn += 1  # False Negative

    # Accuracy
    accuracy = accuracy_score(trues, preds)
    # F1 Score
    f1 = f1_score(trues, preds)
    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    # Sensitivity
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    # MCC
    mcc = matthews_corrcoef(trues, preds)

    # AUC - Ensure probs are in numpy format and length matches
    all_probs_np = np.array(all_probs).flatten()
    trues_np = np.array(trues)
    auc = roc_auc_score(trues_np, all_probs_np)

    print(f"Accuracy: {accuracy:.4f} | F1 Score: {f1:.4f} | Sensitivity (SN): {sensitivity:.4f} | Specificity (SP): {specificity:.4f} | MCC: {mcc:.4f} | AUC: {auc:.4f}")
    print("Confusion Matrix:\n", [[tp, fp], [fn, tn]])

    return accuracy, f1, sensitivity, specificity, mcc, auc

# === STAGE 1: ENHANCER VS NON-ENHANCER ===
def prepare_stage1():
    enhancer_train = load_fasta_clean("/kaggle/input/taha-apu/enhancer.cv.txt")
    non_enhancer_train = load_fasta_clean("/kaggle/input/taha-apu/non.cv.txt")
    
    enhancer_test = load_fasta_clean("/kaggle/input/taha-apu/enhancer.ind.txt")
    non_enhancer_test = load_fasta_clean("/kaggle/input/taha-apu/non.ind.txt")

    min_len_train = min(len(enhancer_train), len(non_enhancer_train))
    enhancer_train = enhancer_train[:min_len_train]
    non_enhancer_train = non_enhancer_train[:min_len_train]

    min_len_test = min(len(enhancer_test), len(non_enhancer_test))
    enhancer_test = enhancer_test[:min_len_test]
    non_enhancer_test = non_enhancer_test[:min_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in non_enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in non_enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test


# === STAGE 2: STRONG VS WEAK ENHANCERS ===
def prepare_stage2():
    strong_train = load_fasta_clean("/kaggle/input/taha-apu/strong_742.txt")
    weak_train = load_fasta_clean("/kaggle/input/taha-apu/weak_742.txt")

    strong_test = load_fasta_clean("/kaggle/input/taha-apu/strong_100.txt")
    weak_test = load_fasta_clean("/kaggle/input/taha-apu/weak_100.txt")

    max_len_train = max(len(strong_train), len(weak_train))
    strong_train = (strong_train * ((max_len_train // len(strong_train)) + 1))[:max_len_train]
    weak_train = (weak_train * ((max_len_train // len(weak_train)) + 1))[:max_len_train]

    max_len_test = max(len(strong_test), len(weak_test))
    strong_test = (strong_test * ((max_len_test // len(strong_test)) + 1))[:max_len_test]
    weak_test = (weak_test * ((max_len_test // len(weak_test)) + 1))[:max_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in strong_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in weak_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in strong_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in weak_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test

def train_stage1():
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage1()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = DNABERT_CNN(dropout_rate=0.3).to(device)  # Using DNA BERT + CNN
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 1 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage1.pth")
            print("✅ Stage 1 model saved.")

def train_stage2():
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage2()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = DNABERT_CNN(dropout_rate=0.3).to(device)  # Using DNA BERT + CNN
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 2 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage2.pth")
            print("✅ Stage 2 model saved.")

# === MAIN ===
if __name__ == "__main__":
    train_stage1()
    train_stage2()


Train Label Distribution: Counter({1: 2968, 0: 2968})
Test Label Distribution: Counter({1: 400, 0: 400})


Stage 1 - Epoch 1: 100%|██████████| 742/742 [09:25<00:00,  1.31it/s]


Epoch 1 Loss: 0.4901
Accuracy: 0.7450 | F1 Score: 0.7488 | Sensitivity (SN): 0.7600 | Specificity (SP): 0.7300 | MCC: 0.4902 | AUC: 0.8224
Confusion Matrix:
 [[304, 108], [96, 292]]
✅ Stage 1 model saved.


Stage 1 - Epoch 2: 100%|██████████| 742/742 [09:25<00:00,  1.31it/s]


Epoch 2 Loss: 0.4275
Accuracy: 0.7588 | F1 Score: 0.7529 | Sensitivity (SN): 0.7350 | Specificity (SP): 0.7825 | MCC: 0.5181 | AUC: 0.8291
Confusion Matrix:
 [[294, 87], [106, 313]]
✅ Stage 1 model saved.


...

Epoch 8 Loss: 0.5408
Accuracy: 0.5580 | F1 Score: 0.4977 | Sensitivity (SN): 0.4380 | Specificity (SP): 0.6780 | MCC: 0.1195 | AUC: 0.6743
Confusion Matrix:
 [[219, 161], [281, 339]]


Stage 2 - Epoch 9: 100%|██████████| 928/928 [11:33<00:00,  1.34it/s]


Epoch 9 Loss: 0.5113
Accuracy: 0.5580 | F1 Score: 0.4908 | Sensitivity (SN): 0.4260 | Specificity (SP): 0.6900 | MCC: 0.1203 | AUC: 0.6708
Confusion Matrix:
 [[213, 155], [287, 345]]


Stage 2 - Epoch 10: 100%|██████████| 928/928 [11:33<00:00,  1.34it/s]


Epoch 10 Loss: 0.4952
Accuracy: 0.6440 | F1 Score: 0.7018 | Sensitivity (SN): 0.8380 | Specificity (SP): 0.4500 | MCC: 0.3125 | AUC: 0.6712
Confusion Matrix:
 [[419, 275], [81, 225]]
✅ Stage 2 model saved.


**DNABERT + Transformer Encoder :**

In [8]:
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, matthews_corrcoef, roc_auc_score
from collections import Counter
import random
import re
from tqdm import tqdm

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Data Utilities ---
def clean_sequence(seq):
    seq = seq.upper()
    return re.sub('[^ATCG]', '', seq)

def load_fasta_clean(file_path):
    with open(file_path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    complement = str.maketrans("ATCG", "TAGC")
    return seq.translate(complement)[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

# --- Shared Dataset Class ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        encoding = self.tokenizer(self.sequences[idx], padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.float),
        }

# --- DNA BERT + Transformer Encoder Model ---
class DNABERT_TransformerEncoder(nn.Module):
    def __init__(self, dropout_rate=0.3, transformer_hidden_size=256, transformer_num_layers=4):
        super().__init__()
        # Load pre-trained DNA BERT model
        self.bert = AutoModel.from_pretrained("zhihan1996/DNA_bert_6")
        
        # Transformer Encoder Layer
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=self.bert.config.hidden_size, 
                                                        nhead=8, 
                                                        dim_feedforward=transformer_hidden_size)
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=transformer_num_layers)
        
        # Fully connected layer
        self.fc1 = nn.Linear(self.bert.config.hidden_size, 1)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self, input_ids, attention_mask):
        # Pass through BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Get the sequence output (not pooler output)
        x = outputs.last_hidden_state
        
        # Pass through Transformer Encoder
        x = self.transformer_encoder(x)
        
        # Use the representation of the last hidden state (CLS token representation)
        x = x[:, 0, :]  # Get first token (CLS token representation)
        
        # Pass through dropout and fully connected layer
        x = self.dropout(x)
        x = self.fc1(x)
        
        return x

# --- Evaluation Utility ---
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    tp, tn, fp, fn = 0, 0, 0, 0  # Initialize confusion matrix counts
    all_probs = []  # List to store all probability values

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)
            logits = model(input_ids, attention_mask).squeeze()
            probs = torch.sigmoid(logits)
            preds.extend((probs > 0.5).int().cpu().tolist())
            trues.extend(label.int().cpu().tolist())
            all_probs.extend(probs.cpu().tolist())  # Collect all probs

            # Update confusion matrix
            for true, pred in zip(label.int().cpu().tolist(), (probs > 0.5).int().cpu().tolist()):
                if true == 1 and pred == 1:
                    tp += 1  # True Positive
                elif true == 0 and pred == 0:
                    tn += 1  # True Negative
                elif true == 0 and pred == 1:
                    fp += 1  # False Positive
                elif true == 1 and pred == 0:
                    fn += 1  # False Negative

    # Accuracy
    accuracy = accuracy_score(trues, preds)
    # F1 Score
    f1 = f1_score(trues, preds)
    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    # Sensitivity
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    # MCC
    mcc = matthews_corrcoef(trues, preds)

    # AUC - Ensure probs are in numpy format and length matches
    all_probs_np = np.array(all_probs).flatten()
    trues_np = np.array(trues)
    auc = roc_auc_score(trues_np, all_probs_np)

    print(f"Accuracy: {accuracy:.4f} | F1 Score: {f1:.4f} | Sensitivity (SN): {sensitivity:.4f} | Specificity (SP): {specificity:.4f} | MCC: {mcc:.4f} | AUC: {auc:.4f}")
    print("Confusion Matrix:\n", [[tp, fp], [fn, tn]])

    return accuracy, f1, sensitivity, specificity, mcc, auc

# === STAGE 1: ENHANCER VS NON-ENHANCER ===
def prepare_stage1():
    enhancer_train = load_fasta_clean("/kaggle/input/taha-apu/enhancer.cv.txt")
    non_enhancer_train = load_fasta_clean("/kaggle/input/taha-apu/non.cv.txt")
    
    enhancer_test = load_fasta_clean("/kaggle/input/taha-apu/enhancer.ind.txt")
    non_enhancer_test = load_fasta_clean("/kaggle/input/taha-apu/non.ind.txt")

    min_len_train = min(len(enhancer_train), len(non_enhancer_train))
    enhancer_train = enhancer_train[:min_len_train]
    non_enhancer_train = non_enhancer_train[:min_len_train]

    min_len_test = min(len(enhancer_test), len(non_enhancer_test))
    enhancer_test = enhancer_test[:min_len_test]
    non_enhancer_test = non_enhancer_test[:min_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in non_enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in non_enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test


# === STAGE 2: STRONG VS WEAK ENHANCERS ===
def prepare_stage2():
    strong_train = load_fasta_clean("/kaggle/input/taha-apu/strong_742.txt")
    weak_train = load_fasta_clean("/kaggle/input/taha-apu/weak_742.txt")

    strong_test = load_fasta_clean("/kaggle/input/taha-apu/strong_100.txt")
    weak_test = load_fasta_clean("/kaggle/input/taha-apu/weak_100.txt")

    max_len_train = max(len(strong_train), len(weak_train))
    strong_train = (strong_train * ((max_len_train // len(strong_train)) + 1))[:max_len_train]
    weak_train = (weak_train * ((max_len_train // len(weak_train)) + 1))[:max_len_train]

    max_len_test = max(len(strong_test), len(weak_test))
    strong_test = (strong_test * ((max_len_test // len(strong_test)) + 1))[:max_len_test]
    weak_test = (weak_test * ((max_len_test // len(weak_test)) + 1))[:max_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in strong_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in weak_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in strong_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in weak_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test

def train_stage1():
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage1()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = DNABERT_TransformerEncoder(dropout_rate=0.3).to(device)  # Using DNA BERT + Transformer Encoder
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 1 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage1.pth")
            print("✅ Stage 1 model saved.")

def train_stage2():
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage2()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = DNABERT_TransformerEncoder(dropout_rate=0.3).to(device)  # Using DNA BERT + Transformer Encoder
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 2 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage2.pth")
            print("✅ Stage 2 model saved.")

# === MAIN ===
if __name__ == "__main__":
    train_stage1()
    train_stage2()


Train Label Distribution: Counter({1: 2968, 0: 2968})
Test Label Distribution: Counter({1: 400, 0: 400})


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Stage 1 - Epoch 1: 100%|██████████| 742/742 [10:47<00:00,  1.15it/s]


Epoch 1 Loss: 0.5268
Accuracy: 0.7675 | F1 Score: 0.7459 | Sensitivity (SN): 0.6825 | Specificity (SP): 0.8525 | MCC: 0.5429 | AUC: 0.8441
Confusion Matrix:
 [[273, 59], [127, 341]]
✅ Stage 1 model saved.


Stage 1 - Epoch 2: 100%|██████████| 742/742 [10:46<00:00,  1.15it/s]


Epoch 2 Loss: 0.4616
Accuracy: 0.7500 | F1 Score: 0.7305 | Sensitivity (SN): 0.6775 | Specificity (SP): 0.8225 | MCC: 0.5053 | AUC: 0.8038
Confusion Matrix:
 [[271, 71], [129, 329]]


...

Epoch 8 Loss: 0.5765
Accuracy: 0.6430 | F1 Score: 0.6982 | Sensitivity (SN): 0.8260 | Specificity (SP): 0.4600 | MCC: 0.3073 | AUC: 0.6329
Confusion Matrix:
 [[413, 270], [87, 230]]
✅ Stage 2 model saved.


Stage 2 - Epoch 9: 100%|██████████| 928/928 [13:08<00:00,  1.18it/s]


Epoch 9 Loss: 0.5512
Accuracy: 0.5620 | F1 Score: 0.5166 | Sensitivity (SN): 0.4680 | Specificity (SP): 0.6560 | MCC: 0.1263 | AUC: 0.6363
Confusion Matrix:
 [[234, 172], [266, 328]]


Stage 2 - Epoch 10: 100%|██████████| 928/928 [13:08<00:00,  1.18it/s]


Epoch 10 Loss: 0.5330
Accuracy: 0.6370 | F1 Score: 0.6566 | Sensitivity (SN): 0.6940 | Specificity (SP): 0.5800 | MCC: 0.2758 | AUC: 0.6429
Confusion Matrix:
 [[347, 210], [153, 290]]


**DNA BERT + BERT :**

In [1]:
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, matthews_corrcoef, roc_auc_score
from collections import Counter
import random
import re
from tqdm import tqdm

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Data Utilities ---
def clean_sequence(seq):
    seq = seq.upper()
    return re.sub('[^ATCG]', '', seq)

def load_fasta_clean(file_path):
    with open(file_path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    complement = str.maketrans("ATCG", "TAGC")
    return seq.translate(complement)[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

# --- Shared Dataset Class ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        encoding = self.tokenizer(self.sequences[idx], padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.float),
        }

# --- DNA BERT + BERT Model ---
class DNABERT_BERT(nn.Module):
    def __init__(self, dropout_rate=0.3):
        super().__init__()
        
        # Load pre-trained DNA BERT
        self.dna_bert = AutoModel.from_pretrained("zhihan1996/DNA_bert_6")
        
        # Load pre-trained BERT
        self.bert = AutoModel.from_pretrained("bert-base-uncased")
        
        # Final classifier layer
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.dna_bert.config.hidden_size + self.bert.config.hidden_size, 1)
        )

    def forward(self, input_ids, attention_mask):
        # Get embeddings from DNA BERT
        dna_bert_outputs = self.dna_bert(input_ids=input_ids, attention_mask=attention_mask)
        dna_bert_emb = dna_bert_outputs.pooler_output
        
        # Get embeddings from BERT
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        bert_emb = bert_outputs.pooler_output
        
        # Concatenate embeddings from both models
        combined_emb = torch.cat((dna_bert_emb, bert_emb), dim=1)
        
        # Pass through classifier
        logits = self.classifier(combined_emb)
        
        return logits

# --- Evaluation Utility ---
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    tp, tn, fp, fn = 0, 0, 0, 0  # Initialize confusion matrix counts
    all_probs = []  # List to store all probability values

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)
            logits = model(input_ids, attention_mask).squeeze()
            probs = torch.sigmoid(logits)
            preds.extend((probs > 0.5).int().cpu().tolist())
            trues.extend(label.int().cpu().tolist())
            all_probs.extend(probs.cpu().tolist())  # Collect all probs

            # Update confusion matrix
            for true, pred in zip(label.int().cpu().tolist(), (probs > 0.5).int().cpu().tolist()):
                if true == 1 and pred == 1:
                    tp += 1  # True Positive
                elif true == 0 and pred == 0:
                    tn += 1  # True Negative
                elif true == 0 and pred == 1:
                    fp += 1  # False Positive
                elif true == 1 and pred == 0:
                    fn += 1  # False Negative

    # Accuracy
    accuracy = accuracy_score(trues, preds)
    # F1 Score
    f1 = f1_score(trues, preds)
    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    # Sensitivity
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    # MCC
    mcc = matthews_corrcoef(trues, preds)

    # AUC - Ensure probs are in numpy format and length matches
    all_probs_np = np.array(all_probs).flatten()
    trues_np = np.array(trues)
    auc = roc_auc_score(trues_np, all_probs_np)

    print(f"Accuracy: {accuracy:.4f} | F1 Score: {f1:.4f} | Sensitivity (SN): {sensitivity:.4f} | Specificity (SP): {specificity:.4f} | MCC: {mcc:.4f} | AUC: {auc:.4f}")
    print("Confusion Matrix:\n", [[tp, fp], [fn, tn]])

    return accuracy, f1, sensitivity, specificity, mcc, auc

# === STAGE 1: ENHANCER VS NON-ENHANCER ===
def prepare_stage1():
    enhancer_train = load_fasta_clean("/kaggle/input/taha-apu/enhancer.cv.txt")
    non_enhancer_train = load_fasta_clean("/kaggle/input/taha-apu/non.cv.txt")
    
    enhancer_test = load_fasta_clean("/kaggle/input/taha-apu/enhancer.ind.txt")
    non_enhancer_test = load_fasta_clean("/kaggle/input/taha-apu/non.ind.txt")

    min_len_train = min(len(enhancer_train), len(non_enhancer_train))
    enhancer_train = enhancer_train[:min_len_train]
    non_enhancer_train = non_enhancer_train[:min_len_train]

    min_len_test = min(len(enhancer_test), len(non_enhancer_test))
    enhancer_test = enhancer_test[:min_len_test]
    non_enhancer_test = non_enhancer_test[:min_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in non_enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in non_enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test


# === STAGE 2: STRONG VS WEAK ENHANCERS ===
def prepare_stage2():
    strong_train = load_fasta_clean("/kaggle/input/taha-apu/strong_742.txt")
    weak_train = load_fasta_clean("/kaggle/input/taha-apu/weak_742.txt")

    strong_test = load_fasta_clean("/kaggle/input/taha-apu/strong_100.txt")
    weak_test = load_fasta_clean("/kaggle/input/taha-apu/weak_100.txt")

    max_len_train = max(len(strong_train), len(weak_train))
    strong_train = (strong_train * ((max_len_train // len(strong_train)) + 1))[:max_len_train]
    weak_train = (weak_train * ((max_len_train // len(weak_train)) + 1))[:max_len_train]

    max_len_test = max(len(strong_test), len(weak_test))
    strong_test = (strong_test * ((max_len_test // len(strong_test)) + 1))[:max_len_test]
    weak_test = (weak_test * ((max_len_test // len(weak_test)) + 1))[:max_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in strong_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in weak_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in strong_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in weak_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test

def train_stage1():
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Check if multiple GPUs are available
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(DNABERT_BERT(dropout_rate=0.3).to(device))
    else:
        model = DNABERT_BERT(dropout_rate=0.3).to(device)

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage1()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 1 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage1.pth")
            print("✅ Stage 1 model saved.")

def train_stage2():
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Check if multiple GPUs are available
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(DNABERT_BERT(dropout_rate=0.3).to(device))
    else:
        model = DNABERT_BERT(dropout_rate=0.3).to(device)

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage2()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 2 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage2.pth")
            print("✅ Stage 2 model saved.")

# === MAIN ===
if __name__ == "__main__":
    train_stage1()
    train_stage2()


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

configuration_bert.py:   0%|          | 0.00/807 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNA_bert_6:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


vocab.txt:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

...

Epoch 8 Loss: 0.5687
Accuracy: 0.6380 | F1 Score: 0.6917 | Sensitivity (SN): 0.8120 | Specificity (SP): 0.4640 | MCC: 0.2944 | AUC: 0.6584
Confusion Matrix:
 [[406, 268], [94, 232]]


Stage 2 - Epoch 9: 100%|██████████| 928/928 [15:20<00:00,  1.01it/s]


Epoch 9 Loss: 0.5441
Accuracy: 0.5460 | F1 Score: 0.4721 | Sensitivity (SN): 0.4060 | Specificity (SP): 0.6860 | MCC: 0.0958 | AUC: 0.6627
Confusion Matrix:
 [[203, 157], [297, 343]]


Stage 2 - Epoch 10: 100%|██████████| 928/928 [15:20<00:00,  1.01it/s]


Epoch 10 Loss: 0.5290
Accuracy: 0.5500 | F1 Score: 0.4944 | Sensitivity (SN): 0.4400 | Specificity (SP): 0.6600 | MCC: 0.1025 | AUC: 0.6647
Confusion Matrix:
 [[220, 170], [280, 330]]


**DNA BERT + CNN + BiLSTM + Attention Mechanism :**

In [1]:
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, matthews_corrcoef, roc_auc_score
from collections import Counter
import random
import re
from tqdm import tqdm

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

# --- Data Utilities ---
def clean_sequence(seq):
    seq = seq.upper()
    return re.sub('[^ATCG]', '', seq)

def load_fasta_clean(file_path):
    with open(file_path) as f:
        lines = f.readlines()
    return [clean_sequence(lines[i+1].strip()) for i in range(0, len(lines), 2)]

def reverse_complement(seq):
    complement = str.maketrans("ATCG", "TAGC")
    return seq.translate(complement)[::-1]

def kmer_tokenize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

# --- Shared Dataset Class ---
class SimpleDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=512):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        encoding = self.tokenizer(self.sequences[idx], padding="max_length", truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.float),
        }

# --- DNA BERT + CNN + BiLSTM + Attention Model ---
class DNABERT_CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, dropout_rate=0.3, cnn_filters=64, lstm_hidden_size=128, attention_heads=8):
        super().__init__()
        # Load pre-trained DNA BERT model
        self.bert = AutoModel.from_pretrained("zhihan1996/DNA_bert_6")
        
        # CNN Layer
        self.conv1 = nn.Conv1d(in_channels=self.bert.config.hidden_size, out_channels=cnn_filters, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(kernel_size=2)  # MaxPooling layer
        
        # BiLSTM Layer
        self.lstm = nn.LSTM(input_size=cnn_filters, hidden_size=lstm_hidden_size, bidirectional=True, batch_first=True)
        
        # Attention Layer
        self.attention = nn.MultiheadAttention(embed_dim=lstm_hidden_size*2, num_heads=attention_heads, batch_first=True)
        
        # Fully connected layer
        self.fc1 = nn.Linear(lstm_hidden_size*2, 1)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self, input_ids, attention_mask):
        # Pass through BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Get the sequence output (not pooler output)
        x = outputs.last_hidden_state
        
        # Apply CNN
        x = x.transpose(1, 2)  # Change shape to (batch_size, channels, sequence_length)
        x = self.conv1(x)      # Apply convolution
        x = self.pool(x)       # Apply max pooling
        x = x.transpose(1, 2)  # Change back shape to (batch_size, sequence_length, channels)
        
        # Apply BiLSTM
        x, _ = self.lstm(x)
        
        # Apply Attention
        attn_output, _ = self.attention(x, x, x)  # Self-attention mechanism
        
        # Use the representation of the last hidden state (CLS token representation)
        x = attn_output[:, 0, :]  # Get first token (CLS token representation)
        
        # Pass through dropout and fully connected layer
        x = self.dropout(x)
        x = self.fc1(x)
        
        return x

# --- Evaluation Utility ---
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    tp, tn, fp, fn = 0, 0, 0, 0  # Initialize confusion matrix counts
    all_probs = []  # List to store all probability values

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)
            logits = model(input_ids, attention_mask).squeeze()
            probs = torch.sigmoid(logits)
            preds.extend((probs > 0.5).int().cpu().tolist())
            trues.extend(label.int().cpu().tolist())
            all_probs.extend(probs.cpu().tolist())  # Collect all probs

            # Update confusion matrix
            for true, pred in zip(label.int().cpu().tolist(), (probs > 0.5).int().cpu().tolist()):
                if true == 1 and pred == 1:
                    tp += 1  # True Positive
                elif true == 0 and pred == 0:
                    tn += 1  # True Negative
                elif true == 0 and pred == 1:
                    fp += 1  # False Positive
                elif true == 1 and pred == 0:
                    fn += 1  # False Negative

    # Accuracy
    accuracy = accuracy_score(trues, preds)
    # F1 Score
    f1 = f1_score(trues, preds)
    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    # Sensitivity
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    # MCC
    mcc = matthews_corrcoef(trues, preds)

    # AUC - Ensure probs are in numpy format and length matches
    all_probs_np = np.array(all_probs).flatten()
    trues_np = np.array(trues)
    auc = roc_auc_score(trues_np, all_probs_np)

    print(f"Accuracy: {accuracy:.4f} | F1 Score: {f1:.4f} | Sensitivity (SN): {sensitivity:.4f} | Specificity (SP): {specificity:.4f} | MCC: {mcc:.4f} | AUC: {auc:.4f}")
    print("Confusion Matrix:\n", [[tp, fp], [fn, tn]])

    return accuracy, f1, sensitivity, specificity, mcc, auc

# === STAGE 1: ENHANCER VS NON-ENHANCER ===
def prepare_stage1():
    enhancer_train = load_fasta_clean("/kaggle/input/taha-apu/enhancer.cv.txt")
    non_enhancer_train = load_fasta_clean("/kaggle/input/taha-apu/non.cv.txt")
    
    enhancer_test = load_fasta_clean("/kaggle/input/taha-apu/enhancer.ind.txt")
    non_enhancer_test = load_fasta_clean("/kaggle/input/taha-apu/non.ind.txt")

    min_len_train = min(len(enhancer_train), len(non_enhancer_train))
    enhancer_train = enhancer_train[:min_len_train]
    non_enhancer_train = non_enhancer_train[:min_len_train]

    min_len_test = min(len(enhancer_test), len(non_enhancer_test))
    enhancer_test = enhancer_test[:min_len_test]
    non_enhancer_test = non_enhancer_test[:min_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in non_enhancer_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in non_enhancer_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test


# === STAGE 2: STRONG VS WEAK ENHANCERS ===
def prepare_stage2():
    strong_train = load_fasta_clean("/kaggle/input/taha-apu/strong_742.txt")
    weak_train = load_fasta_clean("/kaggle/input/taha-apu/weak_742.txt")

    strong_test = load_fasta_clean("/kaggle/input/taha-apu/strong_100.txt")
    weak_test = load_fasta_clean("/kaggle/input/taha-apu/weak_100.txt")

    max_len_train = max(len(strong_train), len(weak_train))
    strong_train = (strong_train * ((max_len_train // len(strong_train)) + 1))[:max_len_train]
    weak_train = (weak_train * ((max_len_train // len(weak_train)) + 1))[:max_len_train]

    max_len_test = max(len(strong_test), len(weak_test))
    strong_test = (strong_test * ((max_len_test // len(strong_test)) + 1))[:max_len_test]
    weak_test = (weak_test * ((max_len_test // len(weak_test)) + 1))[:max_len_test]

    X_train, y_train = [], []
    X_test, y_test = [], []

    for seq in strong_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(1)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(1)
    for seq in weak_train:
        X_train.append(kmer_tokenize(seq))
        y_train.append(0)
        X_train.append(kmer_tokenize(reverse_complement(seq)))
        y_train.append(0)

    for seq in strong_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(1)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(1)
    for seq in weak_test:
        X_test.append(kmer_tokenize(seq))
        y_test.append(0)
        X_test.append(kmer_tokenize(reverse_complement(seq)))
        y_test.append(0)

    return X_train, y_train, X_test, y_test

def train_stage1():
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage1()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = DNABERT_CNN_BiLSTM_Attention(dropout_rate=0.3).to(device)  # Using DNA BERT + CNN + BiLSTM + Attention
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 1 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage1.pth")
            print("✅ Stage 1 model saved.")

def train_stage2():
    tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNA_bert_6", do_lower_case=False, trust_remote_code=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Prepare data
    X_train, y_train, X_test, y_test = prepare_stage2()
    print("Train Label Distribution:", Counter(y_train))
    print("Test Label Distribution:", Counter(y_test))

    train_ds = SimpleDataset(X_train, y_train, tokenizer)
    test_ds = SimpleDataset(X_test, y_test, tokenizer)
    batch_size = 8  # Starting batch size
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    model = DNABERT_CNN_BiLSTM_Attention(dropout_rate=0.3).to(device)  # Using DNA BERT + CNN + BiLSTM + Attention
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5)

    # Learning rate scheduler (Cosine Annealing)
    scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

    best_f1 = 0
    for epoch in range(10):  # 10 epochs
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Stage 2 - Epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            label = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask).squeeze()
            loss = loss_fn(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")
        evaluate(model, test_loader, device)

        # Update learning rate based on scheduler
        scheduler.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                label = batch["label"].to(device)
                logits = model(input_ids, attention_mask).squeeze()
                preds.extend((torch.sigmoid(logits) > 0.5).int().cpu().tolist())
                trues.extend(label.int().cpu().tolist())
        f1 = f1_score(trues, preds)
        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "enhancer_stage2.pth")
            print("✅ Stage 2 model saved.")

# === MAIN ===
if __name__ == "__main__":
    train_stage1()
    train_stage2()


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

configuration_bert.py:   0%|          | 0.00/807 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/zhihan1996/DNA_bert_6:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


vocab.txt:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

...

Epoch 8 Loss: 0.3460
Accuracy: 0.5160 | F1 Score: 0.4873 | Sensitivity (SN): 0.4600 | Specificity (SP): 0.5720 | MCC: 0.0322 | AUC: 0.6242
Confusion Matrix:
 [[230, 214], [270, 286]]


Stage 2 - Epoch 9: 100%|██████████| 928/928 [14:41<00:00,  1.05it/s]


Epoch 9 Loss: 0.3350
Accuracy: 0.6210 | F1 Score: 0.6834 | Sensitivity (SN): 0.8180 | Specificity (SP): 0.4240 | MCC: 0.2633 | AUC: 0.6325
Confusion Matrix:
 [[409, 288], [91, 212]]


Stage 2 - Epoch 10: 100%|██████████| 928/928 [14:41<00:00,  1.05it/s]


Epoch 10 Loss: 0.3218
Accuracy: 0.5350 | F1 Score: 0.4850 | Sensitivity (SN): 0.4380 | Specificity (SP): 0.6320 | MCC: 0.0714 | AUC: 0.6401
Confusion Matrix:
 [[219, 184], [281, 316]]
